# CBMS — Activity Classifier Training

Trains a 2-layer LSTM on MediaPipe Pose keypoint sequences.

**Dataset layout expected:**
```
/kaggle/input/cbms-training-data/
    spitting/    ← short video clips of spitting gesture
    normal/      ← clips of walking, talking, yawning, drinking
    littering/   ← (optional) add more folders to add more classes
```

> After Cell 1: restart session, then run from Cell 2.

## Cell 1 — Install

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "mediapipe==0.10.14", "opencv-python-headless"])
print("Done. Restart session now, then run from Cell 2.")

## Cell 2 — Configuration

In [1]:
import os, json, random, numpy as np
from pathlib import Path

DATA_DIR    = "/kaggle/input/datasets/gallimus/cbms-training-data"
OUTPUT_DIR  = "/kaggle/working/"
MODEL_PATH  = os.path.join(OUTPUT_DIR, "activity_classifier.pt")
LABELS_PATH = os.path.join(OUTPUT_DIR, "class_labels.json")

# Sequence settings
N_FRAMES    = 16      # frames sampled per clip
N_KEYPOINTS = 33      # MediaPipe Pose landmarks
FEATURES    = N_KEYPOINTS * 3   # x, y, z per landmark = 99

# Model architecture
HIDDEN_1    = 128
HIDDEN_2    = 64
DROPOUT     = 0.3

# Training
EPOCHS      = 60
BATCH_SIZE  = 16
LR          = 1e-3
VAL_SPLIT   = 0.20    # 20% of clips used for validation

# Augmentation
AUGMENT     = True    # flip + noise — important for small datasets

# Reproducibility
SEED        = 42
random.seed(SEED)
np.random.seed(SEED)

print("Config loaded.")
print(f"  Input  : {DATA_DIR}")
print(f"  Output : {MODEL_PATH}")
print(f"  Sequence: {N_FRAMES} frames × {FEATURES} features")


Config loaded.
  Input  : /kaggle/input/datasets/gallimus/cbms-training-data
  Output : /kaggle/working/activity_classifier.pt
  Sequence: 16 frames × 99 features


## Cell 3 — Extract pose sequences from video clips

In [2]:
import cv2
import mediapipe as mp

mp_pose = mp.solutions.pose.Pose(
    static_image_mode=True,
    min_detection_confidence=0.3,
)

def extract_sequence(video_path: str, n_frames: int = N_FRAMES) -> np.ndarray | None:
    """
    Sample n_frames evenly from a video clip.
    Returns array of shape (n_frames, FEATURES) or None if extraction fails.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 2:
        cap.release()
        return None

    # Sample frame indices evenly across the clip
    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    sequence = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            sequence.append(np.zeros(FEATURES, dtype=np.float32))
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = mp_pose.process(rgb)

        if result.pose_landmarks:
            kp = result.pose_landmarks.landmark
            # Flatten all 33 landmarks: [x0,y0,z0, x1,y1,z1, ...]
            vec = np.array(
                [[lm.x, lm.y, lm.z] for lm in kp],
                dtype=np.float32
            ).flatten()
        else:
            vec = np.zeros(FEATURES, dtype=np.float32)

        sequence.append(vec)

    cap.release()

    if len(sequence) != n_frames:
        return None

    return np.stack(sequence)   # shape: (n_frames, FEATURES)


def load_dataset(data_dir: str):
    """
    Walk data_dir, extract sequences for each clip.
    Returns (sequences, labels, class_names).
    """
    data_path = Path(data_dir)
    class_names = sorted([
        d.name for d in data_path.iterdir()
        if d.is_dir() and not d.name.startswith('.')
    ])

    if not class_names:
        raise RuntimeError(f"No class subdirectories found in {data_dir}")

    print(f"Classes found: {class_names}")

    sequences, labels = [], []
    vid_exts = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

    for class_idx, class_name in enumerate(class_names):
        class_dir  = data_path / class_name
        clip_files = [
            f for f in class_dir.iterdir()
            if f.suffix.lower() in vid_exts
        ]

        if not clip_files:
            print(f"  WARNING: no clips found in {class_dir}")
            continue

        ok, skip = 0, 0
        for clip_path in clip_files:
            seq = extract_sequence(str(clip_path))
            if seq is None:
                skip += 1
                continue
            sequences.append(seq)
            labels.append(class_idx)
            ok += 1

        print(f"  {class_name:<15} {ok:>3} clips loaded  ({skip} skipped)")

    if not sequences:
        raise RuntimeError("No sequences extracted — check your video files.")

    return (
        np.stack(sequences),      # (N, n_frames, FEATURES)
        np.array(labels),         # (N,)
        class_names
    )


print("Extracting sequences from clips...")
print("(This may take a few minutes depending on how many clips you have)")
print()

X, y, CLASS_NAMES = load_dataset(DATA_DIR)

print(f"\nDataset ready:")
print(f"  X shape     : {X.shape}")
print(f"  y shape     : {y.shape}")
print(f"  Classes     : {CLASS_NAMES}")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:<15}: {(y == i).sum()} clips")


2026-03-22 06:31:11.650348: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774161071.853266     107 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774161071.909189     107 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774161072.318082     107 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774161072.318125     107 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774161072.318128     107 computation_placer.cc:177] computation placer alr

Extracting sequences from clips...
(This may take a few minutes depending on how many clips you have)

Classes found: ['littering', 'normal', 'spitting']


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1774161090.543628     172 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774161090.584810     172 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


  littering        74 clips loaded  (0 skipped)
  normal          161 clips loaded  (0 skipped)
  spitting         38 clips loaded  (0 skipped)

Dataset ready:
  X shape     : (273, 16, 99)
  y shape     : (273,)
  Classes     : ['littering', 'normal', 'spitting']
  littering      : 74 clips
  normal         : 161 clips
  spitting       : 38 clips


## Cell 4 — Augmentation & train/val split

In [3]:
def augment_sequence(seq: np.ndarray) -> np.ndarray:
    """
    Two augmentations:
    1. Mirror flip (x = 1 - x) — simulates person facing other way
    2. Gaussian noise — simulates MediaPipe jitter
    """
    aug = seq.copy().reshape(N_FRAMES, N_KEYPOINTS, 3)
    aug[:, :, 0] = 1.0 - aug[:, :, 0]
    aug += np.random.normal(0, 0.005, aug.shape).astype(np.float32)
    return aug.reshape(N_FRAMES, FEATURES)


def augment_noise_only(seq: np.ndarray) -> np.ndarray:
    """
    Noise-only augmentation (no flip) — used as second pass
    for underrepresented classes to add variety without duplicating
    the mirror-flip version.
    """
    aug = seq.copy().reshape(N_FRAMES, N_KEYPOINTS, 3)
    aug += np.random.normal(0, 0.008, aug.shape).astype(np.float32)
    return aug.reshape(N_FRAMES, FEATURES)


if AUGMENT:
    aug_seqs, aug_labels = [], []

    for seq, label in zip(X, y):
        # Every class gets one mirror flip augmentation
        aug_seqs.append(augment_sequence(seq))
        aug_labels.append(label)

        # Spitting (class index = CLASS_NAMES.index("spitting"))
        # gets a second noise-only pass to partially compensate
        # for having only 38 clips vs littering's 74
        if CLASS_NAMES[label] == "spitting":
            aug_seqs.append(augment_noise_only(seq))
            aug_labels.append(label)

    X = np.concatenate([X, np.stack(aug_seqs)])
    y = np.concatenate([y, np.array(aug_labels)])

    print("After augmentation:")
    for i, name in enumerate(CLASS_NAMES):
        count = (y == i).sum()
        print(f"  {name:<12}: {count} samples")
    print(f"  {'TOTAL':<12}: {len(y)} samples")

# Shuffle before split — critical, do not remove
idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]

# Train / val split
split = int(len(X) * (1 - VAL_SPLIT))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print(f"\nTrain: {len(X_train)}   Val: {len(X_val)}")

After augmentation:
  littering   : 148 samples
  normal      : 322 samples
  spitting    : 114 samples
  TOTAL       : 584 samples

Train: 467   Val: 117


## Cell 5 — Model definition

In [4]:
import torch
import torch.nn as nn

torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


class PoseActivityClassifier(nn.Module):
    """
    2-layer LSTM reading pose keypoint sequences.

    Input:  (batch, n_frames, features)  — e.g. (16, 16, 99)
    Output: (batch, num_classes)         — raw logits, pass through softmax
    """

    def __init__(self, input_size: int, hidden1: int, hidden2: int,
                 num_classes: int, dropout: float):
        super().__init__()

        self.lstm1 = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden1,
            batch_first=True,   # input shape: (batch, seq, features)
        )
        self.lstm2 = nn.LSTM(
            input_size=hidden1,
            hidden_size=hidden2,
            batch_first=True,
        )
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden2, num_classes)

    def forward(self, x):
        # x: (batch, n_frames, features)
        out, _ = self.lstm1(x)          # (batch, n_frames, hidden1)
        out, _ = self.lstm2(out)         # (batch, n_frames, hidden2)
        out     = out[:, -1, :]          # take only final timestep: (batch, hidden2)
        out     = self.dropout(out)
        logits  = self.classifier(out)   # (batch, num_classes)
        return logits


NUM_CLASSES = len(CLASS_NAMES)

model = PoseActivityClassifier(
    input_size=FEATURES,
    hidden1=HIDDEN_1,
    hidden2=HIDDEN_2,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model built — {total_params:,} parameters")
print(f"Classes: {CLASS_NAMES}")


Device: cuda
Model built — 167,107 parameters
Classes: ['littering', 'normal', 'spitting']


## Cell 6 — Training loop

In [5]:
from torch.utils.data import DataLoader, TensorDataset

# Convert to tensors
X_tr = torch.tensor(X_train, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.long)
X_vl = torch.tensor(X_val,   dtype=torch.float32)
y_vl = torch.tensor(y_val,   dtype=torch.long)

train_loader = DataLoader(
    TensorDataset(X_tr, y_tr),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(X_vl, y_vl),
    batch_size=BATCH_SIZE,
)

class_weights = torch.tensor([
    161/74,   # index 0 = littering
    161/161,  # index 1 = normal
    161/38,   # index 2 = spitting  ← now gets the highest weight
], dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=8
)

best_val_acc   = 0.0
best_state     = None
history        = {"train_loss": [], "val_acc": []}

print(f"Training for {EPOCHS} epochs...")
print(f"{'Epoch':>6}  {'Train loss':>11}  {'Val acc':>8}")
print("-" * 34)

for epoch in range(1, EPOCHS + 1):

    # ── Train ────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(xb)
    avg_loss = total_loss / len(X_train)

    # ── Validate ─────────────────────────────────────────────────
    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            preds   = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
    val_acc = correct / len(X_val)

    scheduler.step(val_acc)
    history["train_loss"].append(avg_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    current_lr = optimizer.param_groups[0]["lr"]
    if epoch % 5 == 0:
        print(f"{epoch:>6}  {avg_loss:>11.4f}  {val_acc:>7.1%}  lr={current_lr:.2e}"
              + ("  ← best" if val_acc == best_val_acc else ""))

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.1%}")


Training for 60 epochs...
 Epoch   Train loss   Val acc
----------------------------------
     5       0.9189    51.3%  lr=1.00e-03  ← best
    10       0.8552    62.4%  lr=1.00e-03  ← best
    15       0.6570    53.8%  lr=1.00e-03
    20       0.4123    62.4%  lr=1.00e-03
    25       0.4420    79.5%  lr=1.00e-03
    30       0.3288    87.2%  lr=1.00e-03  ← best
    35       0.2358    88.9%  lr=1.00e-03  ← best
    40       0.1380    88.0%  lr=1.00e-03
    45       0.1234    91.5%  lr=1.00e-03
    50       0.1140    92.3%  lr=1.00e-03
    55       0.0880    88.9%  lr=1.00e-03
    60       0.1281    94.0%  lr=1.00e-03

Training complete. Best val accuracy: 95.7%


## Cell 7 — Save model & labels

In [6]:
# Load best weights back into model
model.load_state_dict(best_state)
model.eval()

# Save checkpoint
torch.save({
    "model_state": best_state,
    "class_names": CLASS_NAMES,
    "n_frames":    N_FRAMES,
    "features":    FEATURES,
    "hidden1":     HIDDEN_1,
    "hidden2":     HIDDEN_2,
    "val_accuracy": best_val_acc,
}, MODEL_PATH)

# Save class labels separately (needed by the pipeline notebook)
with open(LABELS_PATH, "w") as f:
    json.dump(CLASS_NAMES, f)

print(f"Model saved  → {MODEL_PATH}")
print(f"Labels saved → {LABELS_PATH}")
print(f"Val accuracy : {best_val_acc:.1%}")
print(f"Classes      : {CLASS_NAMES}")
print()
print(">>> Download both files from the Output tab <<<")
print(">>> Place them in backend/cv_pipeline/models/ on your local machine <<<")


Model saved  → /kaggle/working/activity_classifier.pt
Labels saved → /kaggle/working/class_labels.json
Val accuracy : 95.7%
Classes      : ['littering', 'normal', 'spitting']

>>> Download both files from the Output tab <<<
>>> Place them in backend/cv_pipeline/models/ on your local machine <<<


## Cell 8 — Evaluation & confusion matrix

In [7]:
import torch
import torch.nn.functional as F

model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for xb, yb in val_loader:
        xb    = xb.to(DEVICE)
        probs = F.softmax(model(xb), dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
        all_preds.extend(preds)
        all_true.extend(yb.numpy())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

# Confusion matrix
print("Confusion matrix (rows=actual, cols=predicted):")
print(f"{'':>12}", end="")
for name in CLASS_NAMES:
    print(f"  {name[:8]:>8}", end="")
print()

for i, true_name in enumerate(CLASS_NAMES):
    print(f"  {true_name[:10]:<10}", end="")
    for j in range(NUM_CLASSES):
        count = ((all_true == i) & (all_preds == j)).sum()
        print(f"  {count:>8}", end="")
    print()

# Per-class accuracy
print()
for i, name in enumerate(CLASS_NAMES):
    mask = all_true == i
    if mask.sum() == 0:
        continue
    acc  = (all_preds[mask] == i).mean()
    print(f"  {name:<15}  accuracy: {acc:.1%}  ({mask.sum()} samples)")

# Overall
overall = (all_preds == all_true).mean()
print(f"\n  Overall accuracy: {overall:.1%}")


Confusion matrix (rows=actual, cols=predicted):
              litterin    normal  spitting
  littering         26         1         1
  normal             0        66         0
  spitting           1         2        20

  littering        accuracy: 92.9%  (28 samples)
  normal           accuracy: 100.0%  (66 samples)
  spitting         accuracy: 87.0%  (23 samples)

  Overall accuracy: 95.7%


## Cell 9 — Single clip inference test
Test the saved model on one clip before integrating it into the pipeline.

In [8]:
def predict_clip(video_path: str, model_path: str) -> dict:
    """
    Run the trained classifier on a single video clip.
    Returns { class_name: probability } dict.
    """
    checkpoint = torch.load(model_path, map_location="cpu")
    names      = checkpoint["class_names"]
    nf         = checkpoint["n_frames"]
    feat       = checkpoint["features"]
    h1         = checkpoint["hidden1"]
    h2         = checkpoint["hidden2"]
    nc         = len(names)

    clf = PoseActivityClassifier(feat, h1, h2, nc, dropout=0.0)
    clf.load_state_dict(checkpoint["model_state"])
    clf.eval()

    seq = extract_sequence(video_path, nf)
    if seq is None:
        return {"error": "could not extract sequence"}

    x      = torch.tensor(seq, dtype=torch.float32).unsqueeze(0)  # (1, n, f)
    with torch.no_grad():
        probs  = torch.softmax(clf(x), dim=1).squeeze().numpy()

    return {name: float(f"{p:.3f}") for name, p in zip(names, probs)}


# ── Test on a clip ────────────────────────────────────────────────────────
# Change this path to any clip from your dataset
TEST_CLIP = "/kaggle/input/datasets/gallimus/cbms-training-data/spitting/VID-20260322-WA0081.mp4"


if os.path.exists(TEST_CLIP):
    result = predict_clip(TEST_CLIP, MODEL_PATH)
    print(f"Predictions for: {TEST_CLIP}")
    for cls, prob in sorted(result.items(), key=lambda x: -x[1]):
        bar = "█" * int(prob * 30)
        print(f"  {cls:<15}  {prob:.1%}  {bar}")
else:
    print(f"Test clip not found: {TEST_CLIP}")
    print("Change TEST_CLIP path to a real clip and re-run.")


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Predictions for: /kaggle/input/datasets/gallimus/cbms-training-data/spitting/VID-20260322-WA0081.mp4
  spitting         99.8%  █████████████████████████████
  normal           0.2%  
  littering        0.0%  


## Cell 10 — Pipeline integration snippet
After downloading `activity_classifier.pt` and `class_labels.json`,
replace the MediaPipe heuristic in `pipeline_v2.ipynb` with this.

In [9]:
# ── Drop-in replacement for extract_activity_features() ──────────────────
#
# In pipeline_v2.ipynb, Cell 6 (helpers):
#   1. Delete extract_activity_features() entirely
#   2. Delete _classify_activity() entirelye
#   3. Add the code below
#   4. In Cell 3 (load models), add:
#        CLASSIFIER, CLASS_NAMES_CLS = load_activity_classifier(
#            "/kaggle/working/activity_classifier.pt"
#        )
#
# In Cell 7 (process_video), the per-track else branch becomes:
#        activity, conf = classify_crop_sequence(
#            list(pre_buffers[tid]), CLASSIFIER, CLASS_NAMES_CLS
#        )
#        if activity != "normal" and conf >= ACTIVITY_CONFIDENCE:
#            capture_state[tid] = {"activity": activity, "frames": list(pre_buffers[tid])}

import torch
import torch.nn.functional as F

def load_activity_classifier(model_path: str):
    ckpt        = torch.load(model_path, map_location="cuda" if torch.cuda.is_available() else "cpu")
    names       = ckpt["class_names"]
    clf         = PoseActivityClassifier(
        ckpt["features"], ckpt["hidden1"], ckpt["hidden2"],
        len(names), dropout=0.0
    ).to(DEVICE)
    clf.load_state_dict(ckpt["model_state"])
    clf.eval()
    print(f"Classifier loaded — classes: {names}")
    return clf, names


def classify_crop_sequence(crops: list, classifier, class_names: list) -> tuple:
    """
    Extract pose keypoints from a list of crops (evidence buffer),
    run the LSTM classifier, return (activity_name, confidence).

    Replaces both extract_activity_features() and _classify_activity().
    Now runs on GPU. No hard-coded rules.
    """
    if len(crops) < 2:
        return "normal", 0.0

    # Sample N_FRAMES evenly from the crops list
    indices = np.linspace(0, len(crops) - 1, N_FRAMES, dtype=int)
    sequence = []

    for idx in indices:
        crop = crops[idx]
        if crop is None or crop.size == 0:
            sequence.append(np.zeros(FEATURES, dtype=np.float32))
            continue

        try:
            rgb    = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            result = MP_POSE.process(rgb)
            if result.pose_landmarks:
                vec = np.array(
                    [[lm.x, lm.y, lm.z]
                     for lm in result.pose_landmarks.landmark],
                    dtype=np.float32
                ).flatten()
            else:
                vec = np.zeros(FEATURES, dtype=np.float32)
        except Exception:
            vec = np.zeros(FEATURES, dtype=np.float32)

        sequence.append(vec)

    seq_arr = np.stack(sequence)                          # (N_FRAMES, FEATURES)
    x       = torch.tensor(seq_arr, dtype=torch.float32)
    x       = x.unsqueeze(0).to(DEVICE)                  # (1, N_FRAMES, FEATURES)

    with torch.no_grad():
        probs = F.softmax(classifier(x), dim=1).cpu().numpy()[0]

    best_idx  = probs.argmax()
    best_name = class_names[best_idx]
    best_conf = float(probs[best_idx])

    # Only return non-normal activities
    if best_name == "normal":
        return "normal", 0.0
    return best_name, round(best_conf, 3)


print("Integration code ready.")
print("See comments above for how to wire this into pipeline_v2.ipynb.")


Integration code ready.
See comments above for how to wire this into pipeline_v2.ipynb.
